[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PatrickJReed/soccer-vision/blob/master/examples/finetune_ball.ipynb)

# Ball detector fine-tune (Phase 2)

Trains a YOLOv8 nano/small model on the labeled Trace ball dataset.
Output: `runs/detect/train/weights/best.pt` — download this and place at
`packages/soccer-vision/src/soccer_vision/models/ball_yolov8_v1.pt` locally.

Run on Colab Pro T4 or L4 GPU. Training takes ~30–60 min.

In [ ]:
!pip install -q "ultralytics>=8.2" roboflow

import os
from google.colab import userdata
os.environ['ROBOFLOW_API_KEY'] = userdata.get('ROBOFLOW_API_KEY')

from pathlib import Path
WORK = Path("/content/work")
WORK.mkdir(exist_ok=True)
%cd /content/work

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key=os.environ['ROBOFLOW_API_KEY'])
# TODO when running: replace with actual workspace / project / version
project = rf.workspace("YOUR_WORKSPACE").project("ball_v1")
dataset = project.version(1).download("yolov8")
print("Dataset at:", dataset.location)

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")  # nano base; small (yolov8s.pt) if recall is weak

results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=100,
    imgsz=1280,
    batch=8,
    patience=20,
    mosaic=1.0,
    mixup=0.15,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    scale=0.5,
    fliplr=0.5,
    project="runs/detect",
    name="ball_v1",
)

In [ ]:
metrics = model.val()
print(f"mAP50: {metrics.box.map50:.3f}")
print(f"Precision: {metrics.box.p[0]:.3f}")
print(f"Recall: {metrics.box.r[0]:.3f}")

In [ ]:
from google.colab import files
files.download("runs/detect/ball_v1/weights/best.pt")